# Welcome

This module is about how generative AI tools work at an intermediate level: broader than the detailed neural network architecture you will be learning in SPC5003 Machine Learning, but considering implementation details that you may have not yet encountered in your use of frontier models.

After completing this module, you will 
* understand common architectures used in generative AI, particularly transformers and diffusers; 
* know how to choose appropriate models, datasets, and fine-tuning methods for given tasks; and hence
* be able to select or develop generative AI models to solve real-world problems.

This first lab is primarily about becoming familiar with Hugging Face, an open repository of model weights.

<div style="border: 1px solid #ccc; padding: 1em;">
<h3>Learning outcomes</h3>

By the end of this lab, you will be able to
<ul>
<li>find suitable open-weight models for a particular task;</li>
<li>download these models from the Hugging Face Hub; and</li>
<li>use the pipelines in the Hugging Face python library to use these models.</li>
</ul>
</div>

## Getting started

▶ Begin by going to [Hugging Face](https://huggingface.co) and, if you haven't already, creating an account for yourself (via the “Sign Up” button at top right).

We’ll come back to the website in a moment, but for now, let’s log in to the Python API so that we can download models directly. 

▶ Execute the code below. Recall from last year that you can do this by putting your cursor in the cell and pressing <kbd>Shift</kbd> + <kbd>Enter</kbd> (or, if you prefer, <kbd>Alt</kbd> + <kbd>Enter</kbd> will also open a new blank cell beneath it). Enter your password when prompted.

In [ ]:
from huggingface_hub import login
login()

When we load models, we’ll need to specify which *device* they should run on: a CPU or (preferably) GPU, or perhaps Apple’s “Metal” (MPS). Boilerplate code to do this is below. 

▶ Execute the code below. If you’re running on our JupyterHub server, this should show that we are using `cuda`, an Nvidia architecture that takes advantage of GPUs. If you’re running at home, this might show something else. I will occasionally indicate where particular labs benefit from more processing power. In general, the transformers we study in the first half of the semester should run perfectly well on CPU, although of course they will be faster if GPUs are available.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f'Using {device.type=}')

## Image generation

Let’s start with a fun task: image generation from text. We will use [Stable Diffusion 1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), which is several years old now, but still impressive, and small enough to be able to download and use with genuinely modest resources (its larger successor, SDXL, is a better generator but is a bit too resource-heavy to run directly on our JupyterHub). 

The Hugging Face Python API makes a series of ”pipelines” available to make it as easy as possible to use models for a particular task. Here we will use `AutoPipelineForText2Image` from the `diffusers` library (since “diffuser” is the general class of model here – we’ll learn more about them from week 8).

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16",
).to(device)

Now let’s pick an image to generate.

In [ ]:
prompt = "a photograph of an astronaut riding a horse"
pipe(prompt).images[0]

▶ Try generating an image from a prompt of your own devising. How does the model cope with unusual or contradictory descriptions?

## Text classification

Not everything in the Hugging Face ecosystem is generative: the Hub and the `transformers` library are really a general home for pretrained models, of which text and image generators are just two kinds. It's worth seeing a small example of a *discriminative* model too, since you'll meet plenty of these alongside the generative ones this semester. Here we'll use [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest), a small RoBERTa-based model fine-tuned to label a piece of text as positive, negative, or neutral.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
)

In [ ]:
classifier("This movie is brilliant!")

▶ Try with a few different prompts. Can you “trick” the classifier? 

*Hint: what if the movie were “terribly” good? Can you find a similar adverb the classifier can’t correctly interpret?*

## Text generation

Now for a generative text model. We'll use [`LiquidAI/LFM2.5-230M`](https://huggingface.co/LiquidAI/LFM2.5-230M), another small model that's designed to run comfortably on modest hardware. Given the start of a sentence, it will predict how that sentence is likely to continue: this is the same basic idea behind every large language model you've used, just at a much smaller scale. We'll look at exactly how this works, one token at a time, very soon.

In [ ]:
from transformers import set_seed

generator = pipeline(
    "text-generation",
    model="LiquidAI/LFM2.5-230M",
    device=device,
)

In [ ]:
# Setting the seed means everyone in the room sees the same generated text
set_seed(10)

prompt = "The best thing about generative AI is"
# max_length=None clears the model's default generation-length setting, which
# would otherwise conflict with max_new_tokens and trigger a warning
generator(prompt, max_new_tokens=30, max_length=None)[0]["generated_text"]

▶ Try a different prompt, or run the same prompt again with a different seed. How sensible are the completions? Do they always make grammatical sense?

## Audio generation

Finally, let's generate a short piece of music from a text description, using [`facebook/musicgen-small`](https://huggingface.co/facebook/musicgen-small), a compact text-to-audio model that's likewise small enough to run on a laptop.

In [ ]:
audio_generator = pipeline(
    "text-to-audio",
    model="facebook/musicgen-small",
    device=device,
)

In [ ]:
data = audio_generator("electric rock solo, very intense")

In [ ]:
import IPython.display as ipd

ipd.Audio(data["audio"], rate=data["sampling_rate"])

▶ Try a different text description of your own devising, perhaps specifying a genre, instrument, or mood. How well does the result match what you asked for?

## Exploring the Hugging Face website

Now return to your web browser and look at the Hugging Face website again.

Across the top of the page, you will see the different sections available. For now let's focus on `Models`. Later we'll also be making substantial use of `Datasets` and `Docs`, among others.

Click on `Models`, then, in the left-hand sidebar, look for `Tasks`. You will see a long list of the different generative-AI and machine-learning tasks that a model might perform. Selecting any of these filters the list of models on the right, sorted by default according to how many times each has been downloaded in the last month, which is a reasonable (though not infallible) proxy for how well-supported and reliable a model is.

Have a browse. Each model's page shows its approximate size, its licence, and often some example usage code you can adapt directly. Not every model is usable with a simple `pipeline()` call in the way we've seen above, and not every model is small enough to download comfortably on a laptop, so it's worth checking both of these before you commit to a download.

### Automatic speech recognition

Let's put this into practice with a task we haven't tried yet: transcribing speech to text. As a sample clip to transcribe, we'll borrow one recording from the [`PolyAI/minds14`](https://huggingface.co/datasets/PolyAI/minds14) dataset, a small collection of people asking a (simulated) bank for help.

In [ ]:
from datasets import load_dataset

sample = load_dataset("PolyAI/minds14", name="en-US", split="train[:1]")[0]
audio = sample["audio"]
ipd.Audio(audio["array"], rate=audio["sampling_rate"])

▶ Go to the Hugging Face `Models` page and filter by the `Automatic Speech Recognition` task. Find a model that looks suitable: not too large to download, with a permissive licence, and with some evidence, from its downloads, likes, or model card, that it performs well on English speech. Load it with `pipeline("automatic-speech-recognition", model=...)` and use it to transcribe `audio`. How does your transcription compare to the ground truth in `sample["transcription"]`?